# Monitoramento de Modelos com MLflow Model Serving

O monitoramento de modelos em produção é fundamental para garantir a qualidade das previsões, identificar problemas de desempenho e detectar possíveis desvios de dados. O MLflow Model Serving oferece uma funcionalidade chamada **tabela de inferência**, que registra informações detalhadas sobre cada requisição feita ao endpoint do modelo, incluindo:

* Payloads de entrada
* Latência de resposta
* Status da inferência (sucesso/erro)
* Resultado da previsão

Neste notebook, vamos demonstrar como habilitar a tabela de inferência, realizar requisições ao endpoint `heart-disease-predict` e analisar as métricas registradas para monitorar o desempenho do modelo em tempo real.

In [0]:
# Habilitando a tabela de inferência no endpoint heart-disease-predict

"""
Para monitorar as requisições feitas ao modelo, é necessário habilitar a tabela de inferência no endpoint MLflow Model Serving. Isso pode ser feito via interface gráfica ou utilizando comandos da API Databricks REST. Abaixo está um exemplo utilizando a API Python do Databricks para habilitar a tabela de inferência.
"""

import requests
import os

# Parâmetros do workspace
workspace_url = "https://dbc-a267a1ed-0d96.cloud.databricks.com/"
api_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
endpoint_name = "heart-disease-predict"

# URL da API para habilitar tabela de inferência
url = f"{workspace_url}/api/2.0/serving-endpoints/{endpoint_name}/inference-table"
headers = {"Authorization": f"Bearer {api_token}"}

# Payload para habilitar a tabela
payload = {"enabled": True}

response = requests.put(url, headers=headers, json=payload)

if response.status_code == 200:
    print("Tabela de inferência habilitada com sucesso!")
else:
    print(f"Erro ao habilitar tabela de inferência: {response.text}")

# Observação: Certifique-se de que as variáveis DATABRICKS_HOST e DATABRICKS_TOKEN estejam configuradas corretamente no ambiente.

In [0]:
# Realizando requisições de exemplo ao endpoint heart-disease-predict

import time
import json

# Exemplo de payload para o modelo de predição de doença cardíaca
payloads = [
    {"age": 63, "sex": 1, "cp": 3, "trestbps": 145, "chol": 233, "fbs": 1, "restecg": 0, "thalach": 150, "exang": 0, "oldpeak": 2.3, "slope": 0, "ca": 0, "thal": 1},
    {"age": 67, "sex": 1, "cp": 2, "trestbps": 160, "chol": 286, "fbs": 0, "restecg": 0, "thalach": 108, "exang": 1, "oldpeak": 1.5, "slope": 1, "ca": 3, "thal": 2},
    {"age": 37, "sex": 1, "cp": 1, "trestbps": 130, "chol": 250, "fbs": 0, "restecg": 1, "thalach": 187, "exang": 0, "oldpeak": 3.0, "slope": 0, "ca": 0, "thal": 1}
]

# Endpoint de inferência
inference_url = f"{workspace_url}/serving-endpoints/{endpoint_name}/invocations"

for i, payload in enumerate(payloads):
    start_time = time.time()
    response = requests.post(
        inference_url,
        headers={"Authorization": f"Bearer {api_token}", "Content-Type": "application/json"},
        data=json.dumps({"data": [payload]})
    )
    latency = time.time() - start_time
    print(f"Requisição {i+1}: status={response.status_code}, latência={latency:.3f}s, resposta={response.text}")

# As requisições acima serão registradas na tabela de inferência do endpoint, permitindo análise posterior.

In [0]:
# Lendo e analisando a tabela de inferência do MLflow Model Serving

import pandas as pd
from pyspark.sql import SparkSession

# Nome genérico da tabela de inferência
inference_table = "mlflow_serving_inference_table"

# Carregar os dados da tabela (substitua pelo nome correto se necessário)
df = spark.table(inference_table)
display(df.limit(10))

# Visualizar as principais colunas disponíveis
print("Colunas da tabela de inferência:", df.columns)

# Exemplo de análise: contar inferências por status
status_counts = df.groupBy("status").count().toPandas()
print(status_counts)

# Exemplo de análise: estatísticas de latência
if "latency_ms" in df.columns:
    latency_stats = df.selectExpr("avg(latency_ms) as avg_latency", "max(latency_ms) as max_latency", "min(latency_ms) as min_latency").toPandas()
    print(latency_stats)
else:
    print("Coluna de latência não encontrada na tabela.")

In [0]:
# Visualizando métricas e gráficos de monitoramento
import matplotlib.pyplot as plt

# Converter para pandas para facilitar a visualização
df_pd = df.toPandas()

# Histograma de latência (se disponível)
if "latency_ms" in df_pd.columns:
    plt.figure(figsize=(8,4))
    plt.hist(df_pd["latency_ms"], bins=20, color='skyblue', edgecolor='black')
    plt.title("Distribuição da Latência das Inferências (ms)")
    plt.xlabel("Latência (ms)")
    plt.ylabel("Frequência")
    plt.show()
else:
    print("Coluna de latência não encontrada na tabela.")

# Gráfico de barras do status das inferências
if "status" in df_pd.columns:
    status_counts = df_pd["status"].value_counts()
    plt.figure(figsize=(6,4))
    status_counts.plot(kind='bar', color='salmon')
    plt.title("Distribuição do Status das Inferências")
    plt.xlabel("Status")
    plt.ylabel("Quantidade")
    plt.show()
else:
    print("Coluna de status não encontrada na tabela.")

# Exemplo: visualizar os primeiros payloads registrados
if "payload" in df_pd.columns:
    print("Exemplo de payloads registrados:")
    print(df_pd["payload"].head())
else:
    print("Coluna de payload não encontrada na tabela.")